# Week 4 — Machine Learning Model Development and Evaluation

Wine Recognition classification with stratified split, Random Forest, GridSearchCV, cross-validation and metrics.

In [ ]:
import numpy as np, pandas as pd
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
wine=load_wine(as_frame=True); df=wine.frame.copy()
df["wine_class"]=df["target"].map({0:"class_0",1:"class_1",2:"class_2"}); df=df.drop(columns="target")
X=df.drop(columns="wine_class"); y=df["wine_class"]
print(df.shape); display(df.head())

## Split data
75% training and 25% testing, stratified to preserve class proportions.

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=42,stratify=y)
print(X_train.shape,X_test.shape)

## Algorithm choice
Random Forest combines multiple decision trees, can capture nonlinear patterns and feature interactions, and does not require feature scaling. Grid search tunes model complexity and ensemble size.

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
pipe=Pipeline([("model",RandomForestClassifier(random_state=42,class_weight="balanced"))])
params={"model__n_estimators":[100,200],"model__max_depth":[None,5,10],"model__min_samples_split":[2,5],"model__min_samples_leaf":[1,2]}
search=GridSearchCV(pipe,params,cv=cv,scoring="accuracy",n_jobs=-1)
search.fit(X_train,y_train)
print("Best parameters:",search.best_params_)
print("Best CV accuracy:",search.best_score_)

## Held-out evaluation

In [ ]:
model=search.best_estimator_; y_pred=model.predict(X_test)
print("Accuracy:",accuracy_score(y_test,y_pred))
print("Macro precision:",precision_score(y_test,y_pred,average="macro"))
print("Macro recall:",recall_score(y_test,y_pred,average="macro"))
print("Macro F1:",f1_score(y_test,y_pred,average="macro"))
print(classification_report(y_test,y_pred))
ConfusionMatrixDisplay.from_estimator(model,X_test,y_test,cmap="Blues"); plt.show()

## Cross-validation and feature importance

In [ ]:
scores=cross_val_score(model,X,y,cv=cv,scoring="accuracy")
print("Fold scores:",scores)
print("Mean:",scores.mean(),"Std:",scores.std())
imp=pd.Series(model.named_steps["model"].feature_importances_,index=X.columns).sort_values()
imp.plot(kind="barh",figsize=(8,6),title="Random Forest Feature Importance"); plt.tight_layout(); plt.show()

## Limitations
This is a small curated benchmark dataset. Cross-validation improves robustness of the estimate but is not external validation. Accuracy is optimized within the specified grid and should not be treated as a guarantee of future performance.